
# 12 — Data Imputation and Encoding

**Scope:** The missing and incorrect values are handled, categorical features are encoded.


Since we decided to predict missing and incorrect values, we have to split the training data into training and validation sets before applying imputation and encoding.

# Table of Contents

Take this as an example for a Table of Contents for your notebook.
We have to fix all the names and sections according to what we actually do in the notebook.

<a class="anchor" id="top"></a>

** **

1. [Importing Libraries & Data](#1.-Importing-Libraries-&-Data) <br><br>
    
2. [Exploratory Data Analysis](#2.-Exploratory-Data-Analysis)
    
   2.1 [Incoherencies](#2.1-Incoherencies) <br>
   
   &emsp; 2.1.1 [Address Identified Incoherencies](#2.1.1-Address-Identified-Incoherencies) <br><br>
    
3. [Data Cleaning & Preprocessing](#3.-Data-Cleaning-&-Preprocessing)

   3.1 [Duplicates](#3.1-Duplicates) <br>
    
   3.2 [Feature Engineering](#3.2-Feature-Engineering) <br>
   
   &emsp; 3.2.1 [Data Type Conversions](#3.2.1-Data-Type-Conversions) <br>
   
   &emsp; 3.2.2 [Encoding](#3.2.2-Encoding) <br>
   
   &emsp; 3.2.3 [Other Transformations](#3.2.3-Other-Transformations) <br>
    
   &emsp; 3.2.4 [Unique Feature-Pair Analysis](#3.2.4-Unique-Feature-Pair-Analysis) <br> 

   3.3 [Train-Test Split](#3.3-Train-Test-Split) <br>
   
   3.4 [Missing Values](#3.4-Missing-Values) <br>
    
   3.5 [Outliers](#3.5-Outliers) <br>

   3.6 [Visualisations](#3.6-Visualisations) <br><br>
   

In [1]:
import os, re, math, warnings
from pathlib import Path
from datetime import datetime
import json
import pandas as pd
import numpy as np
import re


import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("mode.copy_on_write", True)
warnings.filterwarnings("ignore")

RANDOM_STATE = 42  # for reproducibility of any sampling


In [2]:
# Load the data paths
data_dir = "../data/"

# Load the raw data into a pandas dataframe
df = pd.read_csv(os.path.join(data_dir, "processed_data/11_processed_train_data.csv"))
x_test = pd.read_csv(os.path.join(data_dir, "test.csv"))

# Drop the column carID from both dataframes because it is not needed for modeling
if "carID" in df.columns:
    df = df.drop(columns=["carID"])
if "carID" in x_test.columns:
    x_test = x_test.drop(columns=["carID"])


print("Loaded shape:", df.shape)
display(df.head(3))

print("Loaded test shape:", x_test.shape)
display(x_test.head(3))


Loaded shape: (75973, 13)


,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,Volkswagen,Golf,2016.0,22290.0,semi-auto,28421.0,petrol,NaN,11.417268,2.0,63.0,4.0,0.0
1,Toyota,Yaris,2019.0,13790.0,manual,4589.0,petrol,145.0,47.900000,1.5,50.0,1.0,0.0
2,Audi,Q2,2019.0,24990.0,semi-auto,3624.0,petrol,145.0,40.900000,1.5,56.0,4.0,0.0


Loaded test shape: (32567, 12)


,Brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,Hyundai,I30,2022.878006,Automatic,30700.000000,petrol,205.0,41.5,1.6,61.0,3.0,0.0
1,VW,Tiguan,2017.000000,Semi-Auto,-48190.655673,Petrol,150.0,38.2,2.0,60.0,2.0,0.0
2,BMW,2 Series,2016.000000,Automatic,36792.000000,Petrol,125.0,51.4,1.5,94.0,2.0,0.0


### Split the Training data into training and validation sets

In [3]:
# ======================================================
# Split df into Training and Validation Set (60/20/20 total)
# ======================================================

from sklearn.model_selection import train_test_split

# Separate features and target from the training data
X = df.drop(columns=["price"])
y = df["price"]

# Split df (80% of total data) into training and validation
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.25,        # 25% of 80% train = 20% of total
    random_state=42,
)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"y_val shape:   {y_val.shape}")
print(f"X_test shape:  {x_test.shape}")


X_train shape: (56979, 12)
y_train shape: (56979,)
X_val shape:   (18994, 12)
y_val shape:   (18994,)
X_test shape:  (32567, 12)


# Handling Missing Values

- Numerical: fill with the median
- Categorical: fill with the most frequent value

In [4]:
# ======================================================
# Missing-Value-Report (no leakage)
# ======================================================
import pandas as pd

def missing_report(X: pd.DataFrame, name: str) -> pd.DataFrame:
    mv = X.isna().sum()
    mv = mv[mv > 0].sort_values(ascending=False)
    if mv.empty:
        print(f"[{name}] No missing values found. (n_rows={len(X)})")
        return pd.DataFrame(columns=["n_missing", "pct_missing"])
    pct = (mv / len(X) * 100).round(2)
    report = pd.DataFrame({"n_missing": mv, "pct_missing": pct})
    print(f"[{name}] Missing Values (n_rows={len(X)}):")
    display(report)
    return report

# Reports for the splits
mv_train = missing_report(X_train, "X_train")
mv_val   = missing_report(X_val,   "X_val")
mv_test  = missing_report(x_test,  "x_test")

# Optional: warning if columns are missing only in val/test (but not in train)
cols_mv_train = set(mv_train.index) if not mv_train.empty else set()
cols_mv_val   = set(mv_val.index)   if not mv_val.empty   else set()
cols_mv_test  = set(mv_test.index)  if not mv_test.empty  else set()

only_val = cols_mv_val - cols_mv_train
only_test = cols_mv_test - cols_mv_train
if only_val:
    print("⚠️ Columns with missing values only in X_val:", sorted(only_val))
if only_test:
    print("⚠️ Columns with missing values only in x_test:", sorted(only_test))

# Helpful for the next step (imputation/encoding):
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print(f"#numeric_cols: {len(numeric_cols)} → {numeric_cols[:10]}{' ...' if len(numeric_cols) > 10 else ''}")
print(f"#categorical_cols: {len(categorical_cols)} → {categorical_cols[:10]}{' ...' if len(categorical_cols) > 10 else ''}")


[X_train] Missing Values (n_rows=56979):


,n_missing,pct_missing
tax,5978,10.49
mpg,5976,10.49
transmission,2039,3.58
Brand,1144,2.01
engineSize,1144,2.01
previousOwners,1144,2.01
paintQuality%,1141,2.00
hasDamage,1141,2.00
model,1127,1.98
year,1127,1.98


[X_val] Missing Values (n_rows=18994):


,n_missing,pct_missing
mpg,1950,10.27
tax,1926,10.14
transmission,652,3.43
fuelType,416,2.19
hasDamage,407,2.14
previousOwners,406,2.14
model,390,2.05
mileage,385,2.03
paintQuality%,383,2.02
Brand,377,1.98


[x_test] Missing Values (n_rows=32567):


,n_missing,pct_missing
tax,3308,10.16
mpg,3288,10.10
mileage,689,2.12
fuelType,656,2.01
year,653,2.01
model,650,2.00
Brand,649,1.99
engineSize,628,1.93
paintQuality%,625,1.92
transmission,623,1.91


#numeric_cols: 8 → ['year', 'mileage', 'tax', 'mpg', 'engineSize', 'paintQuality%', 'previousOwners', 'hasDamage']
#categorical_cols: 4 → ['Brand', 'model', 'transmission', 'fuelType']


In [5]:
# ======================================================
# Imputation without data leakage (numeric + categorical only, no fallback)
# ======================================================
from sklearn.impute import SimpleImputer
import pandas as pd

# Determine column types from X_train
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

# Imputers (fit ONLY on X_train)
num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

# Numerical: fit/transform
X_train_num = pd.DataFrame(
    num_imputer.fit_transform(X_train[numeric_cols]),
    columns=numeric_cols, index=X_train.index
)
X_val_num = pd.DataFrame(
    num_imputer.transform(X_val[numeric_cols]),
    columns=numeric_cols, index=X_val.index
)
x_test_num = pd.DataFrame(
    num_imputer.transform(x_test[numeric_cols]),
    columns=numeric_cols, index=x_test.index
)

# Categorical: fit/transform (assuming these columns exist)
X_train_cat = pd.DataFrame(
    cat_imputer.fit_transform(X_train[categorical_cols]),
    columns=categorical_cols, index=X_train.index
)
X_val_cat = pd.DataFrame(
    cat_imputer.transform(X_val[categorical_cols]),
    columns=categorical_cols, index=X_val.index
)
x_test_cat = pd.DataFrame(
    cat_imputer.transform(x_test[categorical_cols]),
    columns=categorical_cols, index=x_test.index
)


# Reassemble in the original order
X_train_imp = pd.concat([X_train_num, X_train_cat], axis=1)[X_train.columns]
X_val_imp   = pd.concat([X_val_num,   X_val_cat],   axis=1)[X_val.columns]
x_test_imp  = pd.concat([x_test_num,  x_test_cat],  axis=1)[x_test.columns]

# Optional: quick check
print(
    "NaNs ->",
    "train:", int(X_train_imp.isna().sum().sum()),
    "| val:", int(X_val_imp.isna().sum().sum()),
    "| test:", int(x_test_imp.isna().sum().sum())
)


NaNs -> train: 0 | val: 0 | test: 0


- We have to find out which encoding method is the best for our categorical features

In [9]:
# ======================================================
# Encoding: Target (high-card) + One-Hot (low-card) — leakage-frei
# ======================================================
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from category_encoders.target_encoder import TargetEncoder


high_card_cols = ["Brand", "model"]
low_card_cols  = ["transmission", "fuelType"]

# --- Target Encoding (fit ONLY on X_train, y_train) ---
# Smoothing reduces overfitting; adjust min_samples_leaf/smoothing to the dataset size if needed
te = TargetEncoder(cols=high_card_cols, min_samples_leaf=50, smoothing=10.0)
te.fit(X_train_imp[high_card_cols], y_train)

X_train_te = X_train_imp.copy()
X_val_te   = X_val_imp.copy()
x_test_te  = x_test_imp.copy()

X_train_te[high_card_cols] = te.transform(X_train_imp[high_card_cols])
X_val_te[high_card_cols]   = te.transform(X_val_imp[high_card_cols])
x_test_te[high_card_cols]  = te.transform(x_test_imp[high_card_cols])

# --- One-Hot-Encoding (fit ONLY on X_train) ---
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe.fit(X_train_te[low_card_cols])

def apply_ohe(df: pd.DataFrame) -> pd.DataFrame:
    ohe_mat = ohe.transform(df[low_card_cols])
    ohe_df  = pd.DataFrame(ohe_mat, columns=ohe.get_feature_names_out(low_card_cols), index=df.index)
    # remove the original low-card columns and insert the OHE columns
    out = pd.concat([df.drop(columns=low_card_cols), ohe_df], axis=1)
    # Optionally keep the ordering stable:
    return out

X_train_enc = apply_ohe(X_train_te)
X_val_enc   = apply_ohe(X_val_te)
x_test_enc  = apply_ohe(x_test_te)

# Sanity-Checks
print("Encoded shapes:",
      "train:", X_train_enc.shape,
      "| val:", X_val_enc.shape,
      "| test:", x_test_enc.shape)
print("NaNs after encoding:",
      "train:", int(X_train_enc.isna().sum().sum()),
      "| val:", int(X_val_enc.isna().sum().sum()),
      "| test:", int(x_test_enc.isna().sum().sum()))


Encoded shapes: train: (56979, 20) | val: (18994, 20) | test: (32567, 20)
NaNs after encoding: train: 0 | val: 0 | test: 0


# Scaling

After encoding, all numerical features were scaled using RobustScaler.
This approach reduces the influence of outliers by centering features around the median and scaling them by the interquartile range (IQR).
The scaler was fitted only on the training set to prevent data leakage.
The column carID was dropped since it serves as an identifier and carries no predictive information.


- In the future we can consider scaling the target variable price y
- 

In [10]:
# ======================================================
# 🔢 Feature Scaling (RobustScaler, no data leakage)
# ======================================================
"""
Reasoning:
- Numerical features (e.g., mileage, tax, engineSize) vary widely in scale.
- Since the dataset includes outliers (e.g., price and mileage extremes),
  we use RobustScaler, which scales using the median and IQR instead of mean/std.
- The scaling is fit only on X_train to avoid data leakage.
- carID is dropped because it's only an identifier and not informative.
"""

from sklearn.preprocessing import RobustScaler
import pandas as pd

# 1️⃣ Remove non-informative ID column
drop_cols = ["carID"]
for df_name, df in [("X_train_enc", X_train_enc), ("X_val_enc", X_val_enc), ("x_test_enc", x_test_enc)]:
    if "carID" in df.columns:
        df.drop(columns=drop_cols, inplace=True)
        print(f"Removed {drop_cols} from {df_name}")

# 2️⃣ Update list of numeric columns (exclude ID)
numeric_cols = [c for c in X_train_enc.select_dtypes(include=["number"]).columns if c not in drop_cols]
print("Numeric columns to be scaled:", numeric_cols)

# 3️⃣ Initialize and fit scaler only on training set
scaler = RobustScaler()
scaler.fit(X_train_enc[numeric_cols])

# 4️⃣ Apply scaling to all splits (leakage-free)
X_train_scaled = X_train_enc.copy()
X_val_scaled   = X_val_enc.copy()
x_test_scaled  = x_test_enc.copy()

X_train_scaled[numeric_cols] = scaler.transform(X_train_enc[numeric_cols])
X_val_scaled[numeric_cols]   = scaler.transform(X_val_enc[numeric_cols])
x_test_scaled[numeric_cols]  = scaler.transform(x_test_enc[numeric_cols])

# 5️⃣ Sanity check
print("Scaling complete ✅")
print("Example preview (scaled numeric columns):")
display(X_train_scaled[numeric_cols].head())


Numeric columns to be scaled: ['Brand', 'model', 'year', 'mileage', 'tax', 'mpg', 'engineSize', 'paintQuality%', 'previousOwners', 'hasDamage', 'transmission_automatic', 'transmission_manual', 'transmission_other', 'transmission_semi-auto', 'transmission_unknown', 'fuelType_diesel', 'fuelType_electric', 'fuelType_hybrid', 'fuelType_other', 'fuelType_petrol']
Scaling complete ✅
Example preview (scaled numeric columns):


,Brand,model,year,mileage,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage,transmission_automatic,transmission_manual,transmission_other,transmission_semi-auto,transmission_unknown,fuelType_diesel,fuelType_electric,fuelType_hybrid,fuelType_other,fuelType_petrol
60934,-0.137593,-0.609686,0.000000,0.105590,0.25,0.594406,-0.75,-0.657143,-2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
64124,0.266658,0.097793,-0.333333,0.872986,0.00,0.594406,0.50,0.114286,0.0,0.0,0.0,-1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,-1.0
68043,-0.137593,-0.202712,0.666667,-0.334702,0.00,0.000000,-0.75,0.000000,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
396,-0.137593,0.204450,0.000000,0.440986,0.75,-0.202797,0.50,0.600000,0.0,0.0,0.0,-1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,-1.0
2785,0.862407,0.446106,-0.333333,3.429093,-6.25,1.020979,0.50,0.657143,-0.5,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,-1.0


# Output Save

In [13]:
# ======================================================
# 💾 Save Processed Datasets
# ======================================================
"""
Save X_train, y_train, X_val, y_val, and X_test separately.
This structure is cleaner for later model loading and avoids re-splitting.
"""

import os

output_dir = os.path.join(data_dir, "encoded_data")
os.makedirs(output_dir, exist_ok=True)

# Save feature and target sets separately
X_train_scaled.to_csv(os.path.join(output_dir, "12_X_train.csv"), index=False)
y_train.to_csv(os.path.join(output_dir, "12_y_train.csv"), index=False)

X_val_scaled.to_csv(os.path.join(output_dir, "12_X_val.csv"), index=False)
y_val.to_csv(os.path.join(output_dir, "12_y_val.csv"), index=False)

x_test_scaled.to_csv(os.path.join(output_dir, "12_X_test.csv"), index=False)

print("✅ Processed data saved successfully (X/y separated):")
print(f"X_train: {X_train_scaled.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val_scaled.shape}, y_val: {y_val.shape}")
print(f"X_test:  {x_test_scaled.shape}")


✅ Processed data saved successfully (X/y separated):
X_train: (56979, 20), y_train: (56979,)
X_val:   (18994, 20), y_val: (18994,)
X_test:  (32567, 20)
